In [1]:
# import required libraries for data manipulation and statistics
import pandas as pd
import numpy as np
from scipy import stats

# load pharmacy sales dataset from csv
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

# remove missing records
df = df.dropna().copy()

# calculate daily revenue as units sold multiplied by unit price
df["daily_revenue"] = df["units_sold"] * df["unit_price"]

# list of critical business variables to analyze
variables = ["units_sold", "unit_price", "stock_level", "daily_revenue"]

print("total records loaded:", len(df))
df[variables].head()

total records loaded: 177990


,units_sold,unit_price,stock_level,daily_revenue
0,328,45.38,4774,14884.64
1,371,75.49,4584,28006.79
2,948,22.51,3934,21339.48
3,275,63.29,4544,17404.75
4,563,44.28,4284,24929.64


In [2]:
# calculate parametric 95% confidence intervals
# uses central limit theorem: mean +/- z * (std / sqrt(n))
parametric_results = {}

for col in variables:
    data = df[col].values
    
    # sample size
    n = len(data)
    
    # sample mean and standard error of the mean
    mean = np.mean(data)
    sem = np.std(data, ddof=1) / np.sqrt(n)
    
    # 95% confidence level z-critical value (1.96)
    z_crit = stats.norm.ppf(0.975)
    
    # margin of error
    margin = z_crit * sem
    
    # lower and upper confidence limits
    ci_lower = mean - margin
    ci_upper = mean + margin
    
    parametric_results[col] = {
        "mean": mean,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "margin": margin
    }
    
    # print parametric results for each variable
    print(f"variable: {col}")
    print(f"  sample mean: {mean:.2f}")
    print(f"  parametric 95% ci: [{ci_lower:.2f}, {ci_upper:.2f}] (margin: +/- {margin:.2f})\n")

variable: units_sold
  sample mean: 447.91
  parametric 95% ci: [446.34, 449.49] (margin: +/- 1.57)

variable: unit_price
  sample mean: 52.55
  parametric 95% ci: [52.43, 52.68] (margin: +/- 0.13)

variable: stock_level
  sample mean: 4552.99
  parametric 95% ci: [4551.38, 4554.60] (margin: +/- 1.61)

variable: daily_revenue
  sample mean: 23545.21
  parametric 95% ci: [23435.36, 23655.07] (margin: +/- 109.86)



In [3]:
# calculate bootstrap 95% confidence intervals
# resamples with replacement 1000 times without assuming normal distribution
np.random.seed(42)
bootstrap_results = {}
n_iterations = 1000

for col in variables:
    data = df[col].values
    n = len(data)
    
    # list to store bootstrap means
    boot_means = []
    
    # generate 1000 resamples with replacement
    for _ in range(n_iterations):
        sample = np.random.choice(data, size=n, replace=True)
        boot_means.append(sample.mean())
        
    # calculate 2.5th and 97.5th percentiles for 95% confidence interval
    boot_lower = np.percentile(boot_means, 2.5)
    boot_upper = np.percentile(boot_means, 97.5)
    boot_mean = np.mean(boot_means)
    
    bootstrap_results[col] = {
        "boot_mean": boot_mean,
        "ci_lower": boot_lower,
        "ci_upper": boot_upper
    }
    
    # print bootstrap results for each variable
    print(f"variable: {col}")
    print(f"  bootstrap mean: {boot_mean:.2f}")
    print(f"  bootstrap 95% ci:  [{boot_lower:.2f}, {boot_upper:.2f}]\n")

variable: units_sold
  bootstrap mean: 447.91
  bootstrap 95% ci:  [446.37, 449.45]



variable: unit_price
  bootstrap mean: 52.55
  bootstrap 95% ci:  [52.43, 52.68]



variable: stock_level
  bootstrap mean: 4552.99
  bootstrap 95% ci:  [4551.38, 4554.56]



variable: daily_revenue
  bootstrap mean: 23545.13
  bootstrap 95% ci:  [23440.72, 23664.05]



In [4]:
# create comparison table for parametric vs bootstrap confidence intervals
comparison_rows = []

for col in variables:
    p = parametric_results[col]
    b = bootstrap_results[col]
    
    comparison_rows.append({
        "business variable": col,
        "sample mean": round(p["mean"], 2),
        "parametric 95% ci": f"[{p['ci_lower']:.2f}, {p['ci_upper']:.2f}]",
        "bootstrap 95% ci": f"[{b['ci_lower']:.2f}, {b['ci_upper']:.2f}]",
        "ci width (uncertainty)": round(p["ci_upper"] - p["ci_lower"], 2)
    })

# convert to pandas dataframe for clean display
comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,business variable,sample mean,parametric 95% ci,bootstrap 95% ci,ci width (uncertainty)
0,units_sold,447.91,"[446.34, 449.49]","[446.37, 449.45]",3.15
1,unit_price,52.55,"[52.43, 52.68]","[52.43, 52.68]",0.25
2,stock_level,4552.99,"[4551.38, 4554.60]","[4551.38, 4554.56]",3.22
3,daily_revenue,23545.21,"[23435.36, 23655.07]","[23440.72, 23664.05]",219.71


In [5]:
# business interpretation and decision-making takeaways for pharmacy management
print("--- business takeaways for forecasting and planning ---\n")

# 1. demand forecasting takeaway
print("1. medicine demand (units_sold):")
print(f"   we are 95% confident true average daily demand is between {parametric_results['units_sold']['ci_lower']:.1f} and {parametric_results['units_sold']['ci_upper']:.1f} units.")
print("   planning action: set minimum warehouse replenishment buffer to 450 units per day.\n")

# 2. pricing strategy takeaway
print("2. pricing stability (unit_price):")
print(f"   average medicine price falls tightly between ${parametric_results['unit_price']['ci_lower']:.2f} and ${parametric_results['unit_price']['ci_upper']:.2f}.")
print("   planning action: price variance is very low; standard margin projections are dependable.\n")

# 3. inventory planning takeaway
print("3. stock level management (stock_level):")
print(f"   average inventory on hand is between {parametric_results['stock_level']['ci_lower']:.1f} and {parametric_results['stock_level']['ci_upper']:.1f} units.")
print("   planning action: stock remains comfortably above emergency threshold (4000 units).\n")

# 4. financial budgeting takeaway
print("4. revenue forecasting (daily_revenue):")
print(f"   expected daily revenue per product line falls between ${parametric_results['daily_revenue']['ci_lower']:.2f} and ${parametric_results['daily_revenue']['ci_upper']:.2f}.")
print("   planning action: pharmacy finance department can reliably forecast monthly cash flow using this bounds.")

--- business takeaways for forecasting and planning ---

1. medicine demand (units_sold):
   we are 95% confident true average daily demand is between 446.3 and 449.5 units.
   planning action: set minimum warehouse replenishment buffer to 450 units per day.

2. pricing stability (unit_price):
   average medicine price falls tightly between $52.43 and $52.68.
   planning action: price variance is very low; standard margin projections are dependable.

3. stock level management (stock_level):
   average inventory on hand is between 4551.4 and 4554.6 units.
   planning action: stock remains comfortably above emergency threshold (4000 units).

4. revenue forecasting (daily_revenue):
   expected daily revenue per product line falls between $23435.36 and $23655.07.
   planning action: pharmacy finance department can reliably forecast monthly cash flow using this bounds.
